In [1]:
import re
import time
import itertools
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

### 1. Representation

In [2]:
# literal = (neg, pred, args)
# rule    = (body_literals, head_literal)

VAR_WORDS = {'something', 'someone', 'it', 'they', 'them', 'things', 'people', 'ones', 'one', 'he', 'she'}
AUX = {'is', 'are', 'was', 'were', 'does', 'do', 'can', 'cannot', 'has', 'have', 'did'}
ARTICLES = {'the', 'a', 'an'}

def is_var(t):
    return t[0].isupper()

### 2. Rule-based natural language parser

In [3]:
# Remove articles and identify variables.
def term(words):
    words = [w for w in words if w not in ARTICLES]
    if not words:
        return None
    s = ' '.join(words)
    return 'X' if s in VAR_WORDS else s

# Normalize common verb forms.
def verb_lemma(v):
    if v in ('has', 'have'):
        return 'has'
    return v[:-1] if v.endswith('s') and not v.endswith('ss') else v

# Locate the verb in a tokenized clause.
def find_verb(toks):
    for i, t in enumerate(toks):
        if t in AUX or (i > 0 and toks[i - 1] in VAR_WORDS and i == 1):
            return i
        if i > 0 and t.endswith('s') and not t.endswith('ss') and toks[i - 1] not in ARTICLES and t not in VAR_WORDS:
            return i
    return -1

# Convert a predicate into its negation, name, and arguments.
def parse_pred(subj, toks):
    neg = False
    v = toks[0]
    rest = toks[1:]

    if v == 'cannot':
        return (True, 'can ' + ' '.join(rest), (subj,))

    if rest and rest[0] == 'not':
        neg, rest = True, rest[1:]

    if v in ('does', 'do', 'did'):
        if not rest:
            return None
        v, rest = rest[0], rest[1:]

    if v in ('is', 'are', 'was', 'were'):
        rest = [w for w in rest if w not in ARTICLES]
        if not rest:
            return None
        return (neg, ' '.join(rest), (subj,))

    if v == 'can':
        return (neg, 'can ' + ' '.join(rest), (subj,))

    obj = term(rest)
    if obj is None:
        return (neg, verb_lemma(v), (subj,))
    return (neg, verb_lemma(v), (subj, obj))

# Parse a clause while reusing subjects or verbs when needed.
def parse_clause(text, last_subj=None, last_verb=None):
    toks = text.split()
    if not toks:
        return None, last_subj, last_verb

    if toks[0] == 'not' and last_verb:
        toks = [last_verb] + toks

    vi = find_verb(toks)

    if vi == 0:
        subj = last_subj
    elif vi > 0:
        subj = term(toks[:vi])
    else:
        if last_subj is None or last_verb is None:
            return None, last_subj, last_verb
        subj, toks, vi = last_subj, [last_verb] + toks, 0

    if subj is None:
        return None, last_subj, last_verb

    lit = parse_pred(subj, toks[vi:])
    verb = toks[vi] if toks[vi] in ('is', 'are') else last_verb
    return lit, subj, verb

# Parse a conjunction into individual literals.
def parse_conj(text, last_subj=None):
    lits, verb = [], None
    for part in re.split(r',? and |, ', text):
        lit, last_subj, verb = parse_clause(part.strip(), last_subj, verb)
        if lit is None:
            return None
        lits.append(lit)
    return lits

# Normalize capitalization, punctuation, and contractions.
def norm(s):
    s = s.strip().lower().rstrip('.!?').strip()
    s = s.replace("n't", ' not').replace("isn not", "is not")
    return re.sub(r'\s+', ' ', s)

# Parse a sentence as a rule or a set of facts.
def parse_sentence(s):
    s = norm(s)
    if not s:
        return None

    # Parse conditional rules.
    m = re.match(r'^if (.+?),? then (.+)$', s)
    if m:
        body = parse_conj(m.group(1))
        if body is None:
            return None
        subj = body[0][2][0]
        head = parse_conj(m.group(2), subj)
        if head is None:
            return None
        return [('rule', body, h) for h in head]

    # Parse general rules about categories and properties.
    m = re.match(r'^(?:all )?(.+?) (?:things|people|ones|animals|objects) (?:are|is) (not )?(.+)$', s)
    if m:
        adjs = [a for a in re.split(r',? and |, | ', m.group(1)) if a]
        body = [(False, a, ('X',)) for a in adjs]
        heads = [h for h in re.split(r',? and |, ', m.group(3)) if h]
        return [
            ('rule', body, (bool(m.group(2)), h.replace('a ', '', 1) if h.startswith('a ') else h, ('X',)))
            for h in heads
        ]

    # Parse ordinary statements as facts.
    lits = parse_conj(s)
    if lits is None:
        return None

    out = []
    for lit in lits:
        if any(is_var(a) for a in lit[2]):
            return None
        out.append(('fact', lit))
    return out

# Split text at sentence-ending punctuation.
def split_sentences(text):
    return [x for x in re.split(r'(?<=[.!?])\s+', text.strip()) if x.strip()]

# Separate parsed facts and rules, recording failed sentences.
def parse_kb(context):
    facts, rules, failed = [], [], []

    for s in split_sentences(context):
        p = parse_sentence(s)
        if p is None:
            failed.append(s)
            continue

        for item in p:
            if item[0] == 'fact':
                facts.append(item[1])
            else:
                rules.append((item[1], item[2]))

    return facts, rules, failed


# Parse a query and return its literal if valid.
def parse_query(q):
    p = parse_sentence(q)
    if not p or p[0][0] != 'fact':
        return None
    return p[0][1]

### 3. Forward chaining with unification

In [4]:
# Unify a pattern with a ground tuple and return the variable substitutions.
def unify(pattern, ground, theta):
    theta = dict(theta)
    for p, g in zip(pattern, ground):
        if is_var(p):
            if p in theta and theta[p] != g:
                return None
            theta[p] = g
        elif p != g:
            return None
    return theta


# Apply variable substitutions to predicate arguments.
def subst(args, theta):
    return tuple(theta.get(a, a) for a in args)


# Group rules into strata based on positive and negated predicate dependencies.
def stratify(rules):
    # predicate levels: a rule must run after every predicate it negates
    preds = {r[1][1] for r in rules} | {l[1] for r in rules for l in r[0]}
    level = {p: 0 for p in preds}
    for _ in range(len(preds) + 1):
        changed = False
        for body, head in rules:
            for neg, p, _ in body:
                need = level[p] + (1 if neg else 0)
                if level[head[1]] < need:
                    level[head[1]] = need
                    changed = True
        if not changed:
            break
    else:
        return [rules]          # negation inside a cycle: no stratification, run everything together
    strata = defaultdict(list)
    for r in rules:
        strata[level[r[1][1]]].append(r)
    return [strata[k] for k in sorted(strata)]


# Perform forward chaining to derive facts from rules until a fixpoint is reached.
class ForwardChainer:
    # Initialize the knowledge base, indexes, and derivation tracking.
    def __init__(self, facts, rules, cwa=True):
        self.cwa = cwa
        self.pos, self.neg = set(), set()
        self.why = {}
        for neg, p, a in facts:
            (self.neg if neg else self.pos).add((p, a))
            self.why[(neg, p, a)] = 'given'
        self.by_pred = defaultdict(set)
        for p, a in self.pos:
            self.by_pred[p].add(a)
        self.rules = rules
        self.steps = 0

    # Collect all constants appearing in facts and rules.
    def entities(self):
        return sorted({x for args in list(self.pos) + list(self.neg) for x in args[1]}
                      | {x for b, h in self.rules for l in b + [h] for x in l[2] if not is_var(x)})

    # Check whether a negated predicate holds under the selected assumption.
    def holds_neg(self, p, a):
        return (p, a) not in self.pos if self.cwa else (p, a) in self.neg

    # Match rule body literals against known facts and generate substitutions.
    def matches(self, body, theta):
        if not body:
            yield theta
            return
        lit, rest = body[0], body[1:]
        neg, p, args = lit
        if not neg:
            for g in list(self.by_pred[p]):
                t = unify(args, g, theta)
                if t is not None:
                    yield from self.matches(rest, t)
        else:
            a = subst(args, theta)
            free = [x for x in a if is_var(x)]
            if free:
                if any(not l[0] for l in rest):
                    yield from self.matches(rest + [lit], theta)
                else:
                    for c in self.entities():
                        yield from self.matches([lit] + rest, {**theta, free[0]: c})
                return
            if self.holds_neg(p, a):
                yield from self.matches(rest, theta)

    # Apply rules repeatedly within each stratum until no new facts are derived.
    def run(self):
        for stratum in stratify(self.rules):
            changed = True
            while changed:
                changed = False
                for i, (body, head) in enumerate(stratum):
                    body = sorted(body, key=lambda l: l[0])
                    for theta in list(self.matches(body, {})):
                        self.steps += 1
                        neg, p, args = head
                        a = subst(args, theta)
                        if any(is_var(x) for x in a):
                            continue
                        store = self.neg if neg else self.pos
                        if (p, a) not in store:
                            store.add((p, a))
                            if not neg:
                                self.by_pred[p].add(a)
                            self.why[(neg, p, a)] = (body, head, theta)
                            changed = True
        return self

    # Evaluate a query against the derived facts and negation policy.
    def ask(self, q):
        neg, p, a = q
        if neg:
            return self.holds_neg(p, a)
        return (p, a) in self.pos

    # Explain a query by tracing its supporting facts and rules.
    def trace(self, q, depth=0, shown=None):
        shown = set() if shown is None else shown
        neg, p, a = q
        w = self.why.get(q)
        lit = ('not ' if neg else '') + f"{p}({', '.join(a)})"
        pad = '  ' * depth
        if w is None:
            if neg and self.cwa and self.ask(q):
                return [f"{pad}{lit}  [closed world: {p}({', '.join(a)}) cannot be derived]"]
            return [f"{pad}{lit}  [not derivable]"]
        if w == 'given':
            return [f"{pad}{lit}  [fact]"]
        if q in shown:
            return [f"{pad}{lit}  [derived above]"]
        shown.add(q)
        body, head, theta = w
        lines = [f"{pad}{lit}  [rule: {fmt_rule(body, head)}  with {theta}]"]
        for bn, bp, ba in body:
            lines += self.trace((bn, bp, subst(ba, theta)), depth + 1, shown)
        return lines


# Format a literal as a readable predicate expression.
def fmt_lit(l):
    return ('¬' if l[0] else '') + f"{l[1].replace(' ', '_')}({', '.join(l[2])})"


# Format a rule as a conjunction of premises implying a conclusion.
def fmt_rule(body, head):
    return ' ∧ '.join(fmt_lit(b) for b in body) + ' → ' + fmt_lit(head)

### 4(a) First-order logic with resolution (used for FOLIO)

In [5]:
TOKEN = re.compile(r'\s*(∀|∃|¬|∧|∨|→|↔|⊕|\(|\)|,|=|≠|[^\s(),¬∧∨→↔⊕∀∃=≠]+)')
SYMS = {'&': '∧', '|': '∨', '~': '¬', '!': '¬'}

def tokenize(s):
    s = s.replace('->', '→').replace('<->', '↔').replace('⟷', '↔').replace('⇒', '→')
    toks = [SYMS.get(t, t) for t in TOKEN.findall(s) if t.strip()]
    # repair unbalanced brackets in annotations
    out, depth = [], 0
    for t in toks:
        if t == ')' and depth == 0:
            continue
        depth += (t == '(') - (t == ')')
        out.append(t)
    return out + [')'] * depth


class Parser:
    def __init__(self, s):
        self.t = tokenize(s)
        self.i = 0
        self.bound = []

    def peek(self):
        return self.t[self.i] if self.i < len(self.t) else None

    def eat(self, x=None):
        tok = self.peek()
        if x is not None and tok != x:
            raise SyntaxError(f'expected {x} got {tok}')
        self.i += 1
        return tok

    def parse(self):
        f = self.iff()
        if self.peek() is not None:
            raise SyntaxError('trailing ' + str(self.peek()))
        return f

    def iff(self):
        a = self.imp()
        while self.peek() == '↔':
            self.eat()
            a = ('iff', a, self.imp())
        return a

    def imp(self):
        a = self.disj()
        if self.peek() == '→':
            self.eat()
            return ('imp', a, self.imp())
        return a

    def disj(self):
        a = self.conj()
        while self.peek() in ('∨', '⊕'):
            op = 'or' if self.eat() == '∨' else 'xor'
            a = (op, a, self.conj())
        return a

    def conj(self):
        a = self.unary()
        while self.peek() == '∧':
            self.eat()
            a = ('and', a, self.unary())
        return a

    def unary(self):
        tok = self.peek()
        if tok == '¬':
            self.eat()
            return ('not', self.unary())
        if tok in ('∀', '∃'):
            self.eat()
            v = self.eat()
            self.bound.append(v)
            body = self.unary() if self.peek() in ('(', '∀', '∃', '¬') else self.iff()
            self.bound.remove(v)
            return ('all' if tok == '∀' else 'ex', v, body)
        if tok == '(':
            self.eat('(')
            f = self.iff()
            self.eat(')')
            return f
        return self.atom()

    def term(self):
        name = self.eat()
        if self.peek() == '(':
            self.eat('(')
            args = [self.term()]
            while self.peek() == ',':
                self.eat()
                args.append(self.term())
            self.eat(')')
            return ('fn', name, tuple(args))
        return ('var', name) if name in self.bound else ('const', name.lower())

    def atom(self):
        left = self.term()
        if self.peek() in ('=', '≠'):
            op = self.eat()
            right = self.term()
            f = ('pred', 'Eq', (left, right))
            return ('not', f) if op == '≠' else f
        if left[0] == 'fn':
            return ('pred', left[1], left[2])
        return ('pred', left[1], ())


def parse_fol(s):
    return Parser(s).parse()

In [6]:
def elim(f):
    op = f[0]
    if op == 'pred':
        return f
    if op == 'not':
        return ('not', elim(f[1]))
    if op in ('all', 'ex'):
        return (op, f[1], elim(f[2]))
    a, b = elim(f[1]), elim(f[2])
    if op == 'imp':
        return ('or', ('not', a), b)
    if op == 'iff':
        return ('and', ('or', ('not', a), b), ('or', ('not', b), a))
    if op == 'xor':
        return ('and', ('or', a, b), ('or', ('not', a), ('not', b)))
    return (op, a, b)


def nnf(f):
    op = f[0]
    if op == 'not':
        g = f[1]
        if g[0] == 'pred':
            return f
        if g[0] == 'not':
            return nnf(g[1])
        if g[0] == 'and':
            return ('or', nnf(('not', g[1])), nnf(('not', g[2])))
        if g[0] == 'or':
            return ('and', nnf(('not', g[1])), nnf(('not', g[2])))
        if g[0] == 'all':
            return ('ex', g[1], nnf(('not', g[2])))
        if g[0] == 'ex':
            return ('all', g[1], nnf(('not', g[2])))
    if op in ('and', 'or'):
        return (op, nnf(f[1]), nnf(f[2]))
    if op in ('all', 'ex'):
        return (op, f[1], nnf(f[2]))
    return f


_cnt = itertools.count()


def sub_term(t, m):
    if t[0] == 'var':
        return m.get(t[1], t)
    if t[0] == 'fn':
        return ('fn', t[1], tuple(sub_term(a, m) for a in t[2]))
    return t


def skolem(f, univ, m):
    op = f[0]
    if op == 'pred':
        return ('pred', f[1], tuple(sub_term(a, m) for a in f[2]))
    if op == 'not':
        return ('not', skolem(f[1], univ, m))
    if op in ('and', 'or'):
        return (op, skolem(f[1], univ, m), skolem(f[2], univ, m))
    if op == 'all':
        v = f'V{next(_cnt)}'
        return skolem(f[2], univ + [('var', v)], {**m, f[1]: ('var', v)})
    if op == 'ex':
        name = f'sk{next(_cnt)}'
        t = ('fn', name, tuple(univ)) if univ else ('const', name)
        return skolem(f[2], univ, {**m, f[1]: t})


def to_clauses(f, limit=200):
    # returns list of clauses; clause = frozenset of (neg, pred, args)
    def cnf(g):
        if g[0] == 'and':
            return cnf(g[1]) + cnf(g[2])
        if g[0] == 'or':
            A, B = cnf(g[1]), cnf(g[2])
            if len(A) * len(B) > limit:
                raise ValueError('cnf blowup')
            return [a | b for a in A for b in B]
        if g[0] == 'not':
            p = g[1]
            return [frozenset([(True, p[1], p[2])])]
        return [frozenset([(False, g[1], g[2])])]
    g = skolem(nnf(elim(f)), [], {})
    out = []
    for c in cnf(g):
        if not any((not n, p, a) in c for n, p, a in c):   # drop tautologies
            out.append(c)
    return out

In [7]:
def occurs(v, t, th):
    t = walk(t, th)
    if t == v:
        return True
    return t[0] == 'fn' and any(occurs(v, a, th) for a in t[2])


def walk(t, th):
    while t[0] == 'var' and t in th:
        t = th[t]
    return t


def unify_t(a, b, th):
    a, b = walk(a, th), walk(b, th)
    if a == b:
        return th
    if a[0] == 'var':
        return None if occurs(a, b, th) else {**th, a: b}
    if b[0] == 'var':
        return None if occurs(b, a, th) else {**th, b: a}
    if a[0] == 'fn' and b[0] == 'fn' and a[1] == b[1] and len(a[2]) == len(b[2]):
        for x, y in zip(a[2], b[2]):
            th = unify_t(x, y, th)
            if th is None:
                return None
        return th
    return None


def unify_args(xs, ys, th):
    if len(xs) != len(ys):
        return None
    for x, y in zip(xs, ys):
        th = unify_t(x, y, th)
        if th is None:
            return None
    return th


def resolve_t(t, th):
    t = walk(t, th)
    if t[0] == 'fn':
        return ('fn', t[1], tuple(resolve_t(a, th) for a in t[2]))
    return t


def apply(clause, th):
    return frozenset((n, p, tuple(resolve_t(a, th) for a in args)) for n, p, args in clause)


def rename(clause):
    k = next(_cnt)
    m = {}

    def r(t):
        if t[0] == 'var':
            if t not in m:
                m[t] = ('var', f'R{k}_{len(m)}')
            return m[t]
        if t[0] == 'fn':
            return ('fn', t[1], tuple(r(a) for a in t[2]))
        return t
    return frozenset((n, p, tuple(r(a) for a in args)) for n, p, args in clause)


def term_size(t):
    return 1 + (sum(term_size(a) for a in t[2]) if t[0] == 'fn' else 0)


def clause_weight(c):
    return sum(1 + sum(term_size(a) for a in args) for _, _, args in c)


def resolvents(c1, c2):
    c2 = rename(c2)
    out = []
    for l1 in c1:
        for l2 in c2:
            if l1[0] != l2[0] and l1[1] == l2[1]:
                th = unify_args(l1[2], l2[2], {})
                if th is not None:
                    out.append(apply((c1 - {l1}) | (c2 - {l2}), th))
    return out


def is_taut(c):
    return any((not n, p, a) in c for n, p, a in c)


def resolution_prove(kb_clauses, goal_clauses, max_steps=1000, time_limit=5.0):
    t0 = time.time()
    usable = list(dict.fromkeys(kb_clauses))
    sos = sorted(dict.fromkeys(goal_clauses), key=clause_weight)
    parents = {c: None for c in usable + sos}
    steps = 0
    while sos and steps < max_steps and time.time() - t0 < time_limit:
        given = sos.pop(0)
        usable.append(given)
        for other in usable:
            for r in resolvents(given, other):
                steps += 1
                if r in parents or is_taut(r) or clause_weight(r) > 30:
                    continue
                parents[r] = (given, other)
                if not r:
                    return True, steps, parents
                sos.append(r)
        sos.sort(key=clause_weight)
    return False, steps, parents


def prove_fol(premises_fol, conclusion_fol, **kw):
    kb = []
    for p in premises_fol:
        kb += to_clauses(parse_fol(p))
    goal = to_clauses(('not', parse_fol(conclusion_fol)))
    return resolution_prove(kb, goal, **kw)


def fmt_term(t, names):
    if t[0] == 'fn':
        return f"{t[1]}({', '.join(fmt_term(a, names) for a in t[2])})"
    if t[0] == 'var':
        return names.setdefault(t[1], 'xyzuvw'[len(names) % 6])
    return t[1]


def fmt_clause(c):
    if not c:
        return '□'
    names = {}
    return ' ∨ '.join(('¬' if n else '') + f"{p}({', '.join(fmt_term(a, names) for a in args)})"
                      for n, p, args in sorted(c, key=str))


def proof_lines(parents, c=frozenset(), out=None):
    out = [] if out is None else out
    par = parents.get(c)
    if par:
        for p in par:
            proof_lines(parents, p, out)
        line = f"{fmt_clause(par[0])}  +  {fmt_clause(par[1])}  ⟹  {fmt_clause(c)}"
        if line not in out:
            out.append(line)
    return out

### 4(b) Counterfactual reasoning over a causal model (used for CounterBench)

In [8]:
def parse_causal(context):
    eqs, observed = {}, {}
    text = re.sub(r'we know that', '', context, flags=re.I)
    for sent in re.split(r'\.\s+|\.$', text):
        sent = sent.strip().rstrip('.')
        if not sent or '~' in sent:          
            continue
        m = re.match(r'^We observed (.+)$', sent, re.I)
        if m:
            for part in re.split(r',? and |, ', m.group(1)):
                part = part.strip()
                observed[part[4:] if part.startswith('not ') else part] = not part.startswith('not ')
            continue
        last = None
        for part in re.split(r',\s*', sent):
            part = re.sub(r'^and ', '', part.strip())
            m = re.match(r'^(.+?) (?:together )?causes? (.+)$', part)
            if m:
                lhs, rhs = m.group(1), m.group(2)
                op = 'or' if ' or ' in lhs else 'and'
                parents = [p.strip() for p in re.split(r' and | or ', lhs)]
                last = (op, [(p[4:], True) if p.startswith('not ') else (p, False) for p in parents])
            elif last is None:
                continue
            else:
                rhs = part
            for child in re.split(r' and ', rhs):
                child = child.strip()
                neg = child.startswith('not ')
                child = child[4:] if neg else child
                eqs[child] = (last[0], last[1], neg)
    return eqs, observed


def parse_question(q):
    q = q.strip().rstrip('?')
    factual = {}
    m = re.match(r'^Would (\w+) (?:occur|happen) if (.+)$', q, re.I)
    if m:
        target, cond = m.group(1), m.group(2)
        if ' instead of ' in cond:
            cond, before = cond.split(' instead of ', 1)
            factual = {p[4:] if p.startswith('not ') else p: not p.startswith('not ') for p in re.split(r',? and |, ', before.strip())}
        parts = re.split(r',? and |, ', cond)
    else:
        m = re.search(r'Would (\w+) (?:occur|happen)', q, re.I)
        if not m:
            return None
        target = m.group(1)
        parts = re.findall(r'(?:assume|suppose) (not \w+|\w+)', q, re.I)
    interventions = {}
    for part in parts:
        part = part.strip()
        neg = part.startswith('not ')
        interventions[part[4:] if neg else part] = not neg
    return target, interventions, factual


def evaluate(eqs, roots, do):
    val = dict(roots)
    val.update(do)
    order, done = [], set(val)
    pending = [v for v in eqs if v not in done]
    while pending:
        progress = False
        for v in list(pending):
            op, ps, out_neg = eqs[v]
            if all(p in val for p, _ in ps):
                xs = [val[p] != n for p, n in ps]
                val[v] = (all(xs) if op == 'and' else any(xs)) != out_neg
                pending.remove(v)
                progress = True
        if not progress:
            break
    return val


def counterfactual(eqs, observed, target, do, factual=None, trace=None):
    variables = set(eqs) | {p for _, ps, _ in eqs.values() for p, _ in ps} | set(observed) | set(do)
    roots = sorted(v for v in variables if v not in eqs)
    factual = factual or {}
    for evidence in ({**observed, **factual}, observed, {}):
        answers = []
        for combo in itertools.product([True, False], repeat=len(roots)):
            u = dict(zip(roots, combo))
            world = evaluate(eqs, u, {})
            if any(world.get(k) != v for k, v in evidence.items()):
                continue
            cf = evaluate(eqs, u, do)          # action + prediction
            answers.append(cf.get(target, False))
            if trace is not None and len(trace) < 3:
                trace.append((u, world, cf))
        if answers:
            return sum(answers) * 2 > len(answers)
    return False

### 5. Putting it together: one `predict` function

In [9]:
def is_causal(context, statement):
    return bool(re.search(r'\bcauses?\b', context)) and bool(re.search(r'\bwould \w+ (occur|happen)', statement, re.I))


_kb_cache = {}

def get_chainer(context, cwa=None):
    key = (context, cwa)
    if key not in _kb_cache:
        if len(_kb_cache) > 2000:
            _kb_cache.clear()
        facts, rules, failed = parse_kb(context)
        mode = (not failed) if cwa is None else cwa
        _kb_cache[key] = ForwardChainer(facts, rules, cwa=mode).run()
    return _kb_cache[key]


def predict(context, statement, fol_premises=None, fol_conclusion=None, cwa=None, max_steps=1000):
    if is_causal(context, statement):
        eqs, observed = parse_causal(context)
        q = parse_question(statement)
        return bool(q) and counterfactual(eqs, observed, *q)
    if fol_premises is not None and fol_conclusion:
        try:
            return prove_fol(fol_premises, fol_conclusion, max_steps=max_steps)[0]
        except Exception:
            pass
    q = parse_query(statement)
    if q is None:
        return False
    return get_chainer(context, cwa).ask(q)


print(predict("Bob is young. Bob is kind. If someone is young, then they are smart. "
              "If someone is smart and kind, then they are successful.", "Bob is successful."))

True


### 6. Loading the public datasets

In [10]:
DATA = Path('data')

def read_table(path):
    if path.suffix == '.parquet':
        return pd.read_parquet(path)
    if path.suffix == '.csv':
        return pd.read_csv(path)
    try:
        return pd.read_json(path, lines=path.suffix == '.jsonl')
    except ValueError:
        return pd.read_json(path, lines=True)


def files_in(folder):
    return sorted(p for p in (DATA / folder).rglob('*') if p.suffix in {'.parquet', '.csv', '.json', '.jsonl'})


def to_bool(x):
    return str(x).strip().lower() in {'true', '1', 'yes', 'entailment', 'entailed'}


def load_ruletaker():
    files = files_in('ruletaker')
    test = [f for f in files if 'test' in f.name] or files
    df = pd.concat([read_table(f) for f in test], ignore_index=True)
    q = 'question' if 'question' in df else ('text' if 'text' in df else 'statement')
    return pd.DataFrame({'source': 'ruletaker',
                         'group': df['config'] if 'config' in df else 'all',
                         'context': df['context'], 'statement': df[q],
                         'label': df['label'].map(to_bool)})

def load_counterbench():
    parts = [read_table(f) for f in files_in('counterbench')]
    df = pd.concat([p for p in parts if 'given_info' in p], ignore_index=True)
    return pd.DataFrame({'source': 'counterbench', 'group': df['type'],
                         'context': df['given_info'], 'statement': df['question'],
                         'label': df['answer'].map(to_bool)})


def load_folio():
    parts = []
    for f in files_in('folio'):
        d = read_table(f)
        if 'conclusion-FOL' in d:
            d['group'] = 'validation' if 'valid' in f.name else 'train'
            parts.append(d)
    df = pd.concat(parts, ignore_index=True)
    as_list = lambda x: x if isinstance(x, list) else [s for s in str(x).split('\n') if s.strip()]
    return pd.DataFrame({'source': 'folio', 'group': df['group'],
                         'context': df['premises'].map(lambda x: ' '.join(as_list(x))),
                         'statement': df['conclusion'],
                         'label': df['label'].map(to_bool),
                         'fol_premises': df['premises-FOL'].map(as_list),
                         'fol_conclusion': df['conclusion-FOL']})

data = pd.concat([load_ruletaker(), load_counterbench(), load_folio()], ignore_index=True)
data['id'] = data['source'] + '_' + data.groupby('source').cumcount().astype(str)
data.groupby(['source', 'group']).agg(n=('label', 'size'), true_rate=('label', 'mean')).round(3)

n  true_rate
source       group                               
counterbench backdoor              200      0.475
             basic                 250      0.500
             conditional           250      0.500
             joint                 250      0.500
             nested                250      0.500
folio        train                1001      0.388
             validation            203      0.355
ruletaker    NatLang              7793      0.500
             depth-0             17347      0.514
             depth-1             17788      0.502
             depth-2             17741      0.501
             depth-3             17982      0.501
             depth-3ext          23222      0.503
             depth-3ext-NatLang  31017      0.502
             depth-5             19021      0.500

## 7. Evaluation on the public data
Every query is answered by `predict`, and we time only the parsing + reasoning part.

In [11]:
def run(df, **kw):
    preds, start = [], time.perf_counter()
    for r in df.itertuples():
        fp = r.fol_premises if isinstance(getattr(r, 'fol_premises', None), list) else None
        fc = r.fol_conclusion if isinstance(getattr(r, 'fol_conclusion', None), str) else None
        preds.append(predict(r.context, r.statement, fp, fc, **kw))
    return np.array(preds), time.perf_counter() - start


results = []
data['pred'] = False
for src, df in data.groupby('source', sort=False):
    _kb_cache.clear()
    p, secs = run(df)
    data.loc[df.index, 'pred'] = p
    results.append({'dataset': src, 'queries': len(df),
                    'accuracy (%)': round(100 * (p == df['label']).mean(), 2),
                    'time (s)': round(secs, 2), 'ms / query': round(1000 * secs / len(df), 3)})

summary = pd.DataFrame(results)
total_time = summary['time (s)'].sum()
overall = 100 * (data['pred'] == data['label']).mean()
summary.loc[len(summary)] = ['ALL', len(data), round(overall, 2), round(total_time, 2), round(1000 * total_time / len(data), 3)]
print(f'Final accuracy: {overall:.2f}%   Total inference time: {total_time:.2f} s')
summary

Final accuracy: 99.78%   Total inference time: 40.73 s


,dataset,queries,accuracy (%),time (s),ms / query
0,ruletaker,151911,99.97,10.03,0.066
1,counterbench,1200,94.58,0.12,0.096
2,folio,1204,80.90,30.58,25.398
3,ALL,154315,99.78,40.73,0.264


In [12]:
data['correct'] = data['pred'] == data['label']
data.groupby(['source', 'group']).agg(n=('correct', 'size'), accuracy=('correct', 'mean')).assign(
    accuracy=lambda d: (100 * d['accuracy']).round(2))

n  accuracy
source       group                              
counterbench backdoor              200     90.00
             basic                 250    100.00
             conditional           250     90.00
             joint                 250     96.00
             nested                250     96.00
folio        train                1001     80.42
             validation            203     83.25
ruletaker    NatLang              7793    100.00
             depth-0             17347     99.99
             depth-1             17788    100.00
             depth-2             17741     99.99
             depth-3             17982     99.95
             depth-3ext          23222     99.96
             depth-3ext-NatLang  31017     99.97
             depth-5             19021     99.92

## 8. Experiments
**8.1 How much of the text does the parser understand?**

In [13]:
def coverage(df):
    sents = failed = qfail = 0
    for ctx in df['context'].unique():
        _, _, f = parse_kb(ctx)
        sents += len(split_sentences(ctx))
        failed += len(f)
    qfail = df['statement'].map(lambda s: parse_query(s) is None).sum()
    return {'sentences parsed (%)': round(100 * (1 - failed / sents), 2),
            'queries parsed (%)': round(100 * (1 - qfail / len(df)), 2)}

rt_df = data[data.source == 'ruletaker']
folio_df = data[data.source == 'folio']
cb_df = data[data.source == 'counterbench']
cb_parsed = cb_df['statement'].map(lambda s: parse_question(s) is not None).mean()
pd.DataFrame({'ruletaker': coverage(rt_df), 'folio (NL text)': coverage(folio_df),
              'counterbench': {'sentences parsed (%)': None, 'queries parsed (%)': round(100 * cb_parsed, 2)}}).T

,sentences parsed (%),queries parsed (%)
ruletaker,100.00,100.00
folio (NL text),81.53,80.07
counterbench,NaN,100.00


**8.2 Closed world vs open world on RuleTaker.**

In [14]:
rows = []
for name, mode in [('closed world (CWA)', True), ('open world (OWA)', False)]:
    _kb_cache.clear()
    p, secs = run(rt_df, cwa=mode)
    neg = rt_df['statement'].str.contains(r'\bnot\b').values
    lab = rt_df['label'].values
    rows.append({'mode': name, 'accuracy (%)': round(100 * (p == lab).mean(), 2),
                 'positive queries (%)': round(100 * (p[~neg] == lab[~neg]).mean(), 2),
                 'negated queries (%)': round(100 * (p[neg] == lab[neg]).mean(), 2), 'time (s)': round(secs, 2)})
pd.DataFrame(rows)

,mode,accuracy (%),positive queries (%),negated queries (%),time (s)
0,closed world (CWA),99.97,99.98,99.96,9.67
1,open world (OWA),74.57,94.51,54.84,9.23


**8.3 FOLIO: natural-language parser vs FOL annotations, and the resolution budget.**

In [15]:
_kb_cache.clear()
start = time.perf_counter()
nl_pred = np.array([predict(r.context, r.statement) for r in folio_df.itertuples()])
rows = [{'setting': 'NL text only (forward chaining)', 'accuracy (%)': round(100 * (nl_pred == folio_df['label']).mean(), 2),
         'time (s)': round(time.perf_counter() - start, 2)}]
for steps in [50, 200, 1000, 3000]:
    p, secs = run(folio_df, max_steps=steps)
    rows.append({'setting': f'FOL + resolution, max {steps} steps', 'accuracy (%)': round(100 * (p == folio_df['label']).mean(), 2),
                 'time (s)': round(secs, 2)})
rows.append({'setting': 'always predict False (baseline)', 'accuracy (%)': round(100 * (~folio_df['label']).mean(), 2), 'time (s)': 0})
pd.DataFrame(rows)

,setting,accuracy (%),time (s)
0,NL text only (forward chaining),62.04,0.09
1,"FOL + resolution, max 50 steps",79.82,1.52
2,"FOL + resolution, max 200 steps",80.73,3.63
3,"FOL + resolution, max 1000 steps",80.81,31.78
4,"FOL + resolution, max 3000 steps",80.90,148.29
5,always predict False (baseline),61.79,0.00


**8.4 Where do the remaining errors come from?**

In [16]:
err = data[~data['correct']]
print(pd.crosstab([err['source'], err['label']], err['pred'], rownames=['source', 'label'], colnames=['pred']))
for r in err.groupby('source').head(2).itertuples():
    print('\n[', r.source, '] label =', r.label, '| pred =', r.pred)
    print('context  :', r.context[:300])
    print('statement:', r.statement)
    if r.source == 'folio':
        print('FOL      :', r.fol_conclusion)

pred                False  True 
source       label              
counterbench False      0     20
             True      45      0
folio        False      0     26
             True     204      0
ruletaker    False      0     18
             True      28      0

[ ruletaker ] label = True | pred = False
context  : The bear does not see the lion. The cow sees the lion. The lion eats the bear. The lion is round. The lion sees the cow. The rabbit eats the bear. The rabbit likes the lion. If someone sees the lion and the lion eats the cow then the cow does not see the lion. If the bear eats the rabbit then the ra
statement: The bear does not like the bear.

[ ruletaker ] label = False | pred = True
context  : The bear does not see the lion. The cow sees the lion. The lion eats the bear. The lion is round. The lion sees the cow. The rabbit eats the bear. The rabbit likes the lion. If someone sees the lion and the lion eats the cow then the cow does not see the lion. If the bear eats the r

## 9. Reasoning traces
**Forward chaining (RuleTaker).**

In [17]:
pos = rt_df[(rt_df['group'] == 'depth-5') & rt_df['label']].head(3000)
best = max(pos.itertuples(), key=lambda r: len(get_chainer(r.context).trace(parse_query(r.statement))))
fc = get_chainer(best.context)
print('Context:', best.context, '\n\nQuery:', best.statement, '->', fc.ask(parse_query(best.statement)), '\n')
print('\n'.join(fc.trace(parse_query(best.statement))))

Context: The cat chases the rabbit. The cat is big. The cat is green. The cat is kind. The cat is rough. The cat visits the mouse. The cat visits the rabbit. The mouse is big. The mouse likes the squirrel. The mouse visits the cat. The mouse visits the squirrel. The rabbit chases the squirrel. The rabbit likes the cat. The rabbit visits the squirrel. The squirrel chases the cat. The squirrel visits the rabbit. If someone chases the squirrel and they chase the cat then the cat likes the rabbit. If someone is rough and kind then they chase the squirrel. If someone is green and they visit the mouse then the mouse is cold. If someone is cold and they visit the cat then the cat likes the mouse. If someone visits the mouse and the mouse likes the squirrel then they like the squirrel. If someone is kind and they like the rabbit then the rabbit visits the cat. If someone likes the cat and they chase the mouse then the mouse likes the squirrel. If someone is rough and they like the mouse then t

In [18]:
neg = rt_df[rt_df['label'] & rt_df['statement'].str.contains(' not ')].head(3000)
ex = next(r for r in neg.itertuples() if get_chainer(r.context).why.get(parse_query(r.statement)) not in (None, 'given'))
fc = get_chainer(ex.context)
print('Context:', ex.context, '\n\nQuery:', ex.statement, '->', fc.ask(parse_query(ex.statement)), '\n')
print('\n'.join(fc.trace(parse_query(ex.statement))))

Context: Bob is kind. Bob is not rough. Harry is rough. All quiet, rough people are kind. If someone is not red then they are not quiet. 

Query: Bob is not quiet. -> True 

not quiet(bob)  [rule: ¬red(X) → ¬quiet(X)  with {'X': 'bob'}]
  not red(bob)  [closed world: red(bob) cannot be derived]


**Resolution refutation (FOLIO).**

In [19]:
for r in folio_df[folio_df['label'] & folio_df['pred']].itertuples():
    ok, steps, parents = prove_fol(r.fol_premises, r.fol_conclusion, max_steps=1000)
    lines = proof_lines(parents)
    if 2 <= len(lines) <= 5:
        break
print('Premises (FOL):'); print('\n'.join('  ' + p for p in r.fol_premises))
print('Conclusion:', r.fol_conclusion, '\nNegated goal clauses:', [fmt_clause(c) for c in to_clauses(('not', parse_fol(r.fol_conclusion)))])
print(f'\nProof found in {steps} resolution steps:')
print('\n'.join('  ' + l for l in lines))

Premises (FOL):
  ∀x (DrinkRegularly(x, coffee) → IsDependentOn(x, caffeine))
  ∀x (DrinkRegularly(x, coffee)  ∨ (¬WantToBeAddictedTo(x, caffeine)))
  ∀x (¬WantToBeAddictedTo(x, caffeine) → ¬AwareThatDrug(x, caffeine))
  ¬(Student(rina) ⊕  ¬AwareThatDrug(rina, caffeine))
  ¬(IsDependentOn(rina, caffeine) ⊕ Student(rina))
Conclusion: ¬WantToBeAddictedTo(rina, caffeine) ∨ (¬AwareThatDrug(rina, caffeine)) 
Negated goal clauses: ['WantToBeAddictedTo(rina, caffeine)', 'AwareThatDrug(rina, caffeine)']

Proof found in 10 resolution steps:
  WantToBeAddictedTo(rina, caffeine)  +  DrinkRegularly(x, coffee) ∨ ¬WantToBeAddictedTo(x, caffeine)  ⟹  DrinkRegularly(rina, coffee)
  DrinkRegularly(rina, coffee)  +  IsDependentOn(x, caffeine) ∨ ¬DrinkRegularly(x, coffee)  ⟹  IsDependentOn(rina, caffeine)
  AwareThatDrug(rina, caffeine)  +  ¬AwareThatDrug(rina, caffeine) ∨ ¬Student(rina)  ⟹  ¬Student(rina)
  ¬Student(rina)  +  Student(rina) ∨ ¬IsDependentOn(rina, caffeine)  ⟹  ¬IsDependentOn(rina, caffei

**Counterfactual search (CounterBench).**

In [20]:
ex = cb_df[cb_df['context'].str.contains('observed')].iloc[0]
eqs, observed = parse_causal(ex['context'])
target, do, factual = parse_question(ex['statement'])
tr = []
ans = counterfactual(eqs, observed, target, do, factual, trace=tr)
print(ex['context'], '\n', ex['statement'], '| gold:', ex['label'], '| predicted:', ans, '\n')
for v, (op, ps, out_neg) in eqs.items():
    rhs = f' {op.upper()} '.join(('NOT ' if n else '') + p for p, n in ps)
    print(f'  {v} = ' + (f'NOT ({rhs})' if out_neg else rhs))
print('  observed:', observed, '| factual (instead of):', factual, '| intervention:', do)
for u, world, cf in tr:
    print('\n  exogenous :', u)
    print('  factual   :', world)
    print('  after do():', cf)

We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune 
 Would Lumbo occur if not Ziklo instead of Ziklo? | gold: True | predicted: True 

  Blaf = NOT (Ziklo)
  Vork = Blaf AND Trune
  Lumbo = Vork
  observed: {'Trune': True} | factual (instead of): {'Ziklo': True} | intervention: {'Ziklo': False}

  exogenous : {'Trune': True, 'Ziklo': True}
  factual   : {'Trune': True, 'Ziklo': True, 'Blaf': False, 'Vork': False, 'Lumbo': False}
  after do(): {'Trune': True, 'Ziklo': False, 'Blaf': True, 'Vork': True, 'Lumbo': True}


## 10. Submission file
If the private test file is placed at `data/test.csv` (columns `id, context, statement`) it is used,
otherwise the file is produced for the public evaluation queries above.

In [21]:
test_path = DATA / 'test.csv'
if test_path.exists():
    test = pd.read_csv(test_path)
    _kb_cache.clear()
    start = time.perf_counter()
    test['label'] = [predict(c, s) for c, s in zip(test['context'], test['statement'])]
    print(f'Inference time on test: {time.perf_counter() - start:.2f} s')
    sub = test[['id', 'label']]
else:
    sub = data[['id', 'pred']].rename(columns={'pred': 'label'})
sub.to_csv('submission.csv', index=False)
print(sub.shape)
sub.head()

(154315, 2)


,id,label
0,ruletaker_0,True
1,ruletaker_1,False
2,ruletaker_2,True
3,ruletaker_3,False
4,ruletaker_4,True
